<a href="https://colab.research.google.com/github/RMDilshanTharindu/DilshanGPT_v2/blob/main/DilshanGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install transformers sentence-transformers langchain faiss-cpu langchain-text-splitters

In [75]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
import numpy as np
import os

In [150]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # can upgrade later
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
chatbot = pipeline("text-generation", model=model, tokenizer=tokenizer)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [151]:
chat_history = []
user_profile = {}

In [152]:
docs = []

# Example: upload txt files to Colab
doc_folder = "docs"
os.makedirs(doc_folder, exist_ok=True)

for filename in os.listdir(doc_folder):
    if filename.endswith(".txt"):
        with open(os.path.join(doc_folder, filename), "r", encoding="utf-8") as f:
            docs.append(f.read())

# Split documents
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = []
for doc in docs:
    chunks.extend(splitter.split_text(doc))

# Embed chunks
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = [embed_model.encode(c) for c in chunks]

# Build FAISS index
dimension = embeddings[0].shape[0]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [153]:
def retrieve_docs(query, k=5):
    query_lower = query.lower()

    # Step 1: Keyword filter (STRONG SIGNAL)
    keyword_matches = []
    for c in chunks:
        if any(word in c.lower() for word in query_lower.split()):
            keyword_matches.append(c)

    # Step 2: Semantic search
    query_vec = embed_model.encode([query])
    distances, indices = index.search(np.array(query_vec), k)
    semantic_matches = [chunks[i] for i in indices[0]]

    # Combine (keyword first = priority)
    results = keyword_matches + semantic_matches

    # Remove duplicates
    results = list(dict.fromkeys(results))

    return results[:3]

In [154]:
def chat_with_rag(user_input):
    global chat_history, user_profile

    user_lower = user_input.lower()

    # 🔥 Hard rules
    if "who made you" in user_lower or "who created you" in user_lower:
        return "Dilshan Tharindu, an Undergraduate at USJP."
    if "interest" in user_lower or "interested in" in user_lower:
        return "Dilshan Tharindu is interested in AI and Cybersecurity."
    if "my name is" in user_lower:
        name = user_input.split("is")[-1].strip()
        user_profile["name"] = name
        return f"Nice to meet you, {name}!"
    if "what is my name" in user_lower:
        return f"Your name is {user_profile.get('name', 'unknown')}."

    # Add user message to memory
    chat_history.append(f"<|user|>\n{user_input}\n</s>")
    chat_history = chat_history[-6:]  # keep last 6 exchanges
    history_text = "\n".join(chat_history)

    # Retrieve relevant docs
    retrieved = retrieve_docs(user_input)
    #print("Debug: retreved",retrieved)
    context = "\n".join(retrieved)

    # Build prompt
    prompt = f"""<|system|>
    You are DilshanGPT.

    STRICT RULES:
    - Only answer using given context or known facts below
    - Do NOT make up information
    - If unknown, say "I don't know"
    - Give only The direct answer to the question
    - Do NOT explain
    - Do NOT list multiple points
    - Answer in ONE short sentence

    Known facts:
    - Created by Dilshan Tharindu
    - Born in 2003 in Sri Lanka
    - Interested in AI and Cybersecurity

    Context:
    {context}

    Conversation:
    {history_text}

    User: {user_input}
    Assistant:
    """

    response = chatbot(
        prompt,
        max_new_tokens=150,
        do_sample=False,
        return_full_text=False
    )
    output = response[0]["generated_text"]
    answer = output.split("<|assistant|>")[-1].strip()

    # Save assistant response to memory
    chat_history.append(f"<|assistant|>\n{answer}\n</s>")

    answer = response[0]["generated_text"].strip()

    # Keep only first line
    answer = answer.split("\n")[0]

    # Remove numbering like "1."
    if answer.startswith("1."):
        answer = answer[2:].strip()

    return answer

In [155]:
#print(chat_with_rag("Hi"))
#print(chat_with_rag("My name is Kaweesha"))
#print(chat_with_rag("What is my name?"))
#print(chat_with_rag("Who created you?"))
#print(chat_with_rag("what is Dilshan's hidden truth?"))
print(chat_with_rag("What is Dilshan Tharindu's secret is?"))
#print(chat_with_rag("How dilshan Tharindu can be contacted"))
#print(chat_with_rag("How dilshan Tharindu can be contacted through a phone "))
#print(chat_with_rag("what is Tharindu phone number "))



Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I love AI


In [156]:
import gradio as gr

def chat_ui(user_input):
    return chat_with_rag(user_input)

iface = gr.Interface(
    fn=chat_ui,
    inputs=gr.Textbox(lines=2, placeholder="Ask DilshanGPT anything..."),
    outputs=gr.Textbox(lines=5, placeholder="DilshanGPT replies here..."),
    title="DilshanGPT v4 🔥",
    description="RAG + TinyLlama AI assistant created by Dilshan Tharindu"
)

iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://91fc4d6f8e9bc400f2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
